In [ ]:
import os
import cv2
import glob
import datetime
import time as tm
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import keras
import random

from sklearn.model_selection  import train_test_split
from sklearn.metrics import roc_auc_score
from keras.utils import to_categorical
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy("mixed_float16")
# tf.config.optimizer.set_jit(True)

print(tf.__version__)
print(keras.__version__)
print(np.__version__)
print(tf.keras.__version__)

In [ ]:
# SEED = 42
# def set_seeds(seed=SEED):
#     os.environ['PYTHONHASHSEED'] = str(seed)
#     random.seed(seed)
#     tf.random.set_seed(seed)
#     np.random.seed(seed)

# def set_global_determinism(seed=SEED):
#     set_seeds(seed=seed)

#     os.environ['TF_DETERMINISTIC_OPS'] = '1'
#     os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    
#     tf.config.threading.set_inter_op_parallelism_threads(1)
#     tf.config.threading.set_intra_op_parallelism_threads(1)

# # Call the above function with seed value
# set_global_determinism(seed=SEED)

In [ ]:
# img_size1 = 512
# img_size2 = 256
# payload = 20

# X_train = np.load(f'../input/suni-2/SUNI/BossSUNI{img_size1}x{img_size2}x{payload}-X_train.npy')
# X_valid = np.load(f'../input/suni-2/SUNI/BossSUNI{img_size1}x{img_size2}x{payload}-X_valid.npy')
# X_test = np.load(f'../input/suni-2/SUNI/BossSUNI{img_size1}x{img_size2}x{payload}-X_test.npy')
# y_train = np.load(f'../input/suni-2/SUNI/BossSUNI{img_size1}x{img_size2}x{payload}-y_train.npy')
# y_valid = np.load(f'../input/suni-2/SUNI/BossSUNI{img_size1}x{img_size2}x{payload}-y_valid.npy')
# y_test = np.load(f'../input/suni-2/SUNI/BossSUNI{img_size1}x{img_size2}x{payload}-y_test.npy')
# # --- Output shapes ---
# print("Train:", X_train.shape, y_train.shape)
# print("Valid:", X_valid.shape, y_valid.shape)
# print("Test:",  X_test.shape,  y_test.shape)

In [ ]:
# img_size1 = 256
# img_size2 = 256
# payload = 20

# X_train = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-X_train.npy')
# X_valid = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-X_valid.npy')
# X_test = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-X_test.npy')
# y_train = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-y_train.npy')
# y_valid = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-y_valid.npy')
# y_test = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-y_test.npy')
# # --- Output shapes ---
# print("Train:", X_train.shape, y_train.shape)
# print("Valid:", X_valid.shape, y_valid.shape)
# print("Test:",  X_test.shape,  y_test.shape)

In [ ]:
img_size1 = 256
img_size2 = 256
payload = 50

X_train = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-X_train.npy')
X_valid = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-X_valid.npy')
X_test = np.load(f'../input/alsk-csnpy/alsk-mipod/alsk-mipod-X_train.npy')
y_train = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-y_train.npy')
y_valid = np.load(f'../input/mipod-npy/BossMIPOD{img_size1}x{img_size2}x{payload}-y_valid.npy')
y_test = np.load(f'../input/alsk-csnpy/alsk-mipod/alsk-mipod-y_train.npy')
# --- Output shapes ---
print("Train:", X_train.shape, y_train.shape)
print("Valid:", X_valid.shape, y_valid.shape)
print("Test:",  X_test.shape,  y_test.shape)

In [ ]:
from tensorflow.data import AUTOTUNE

# Setting Architecture
MODEL_SIZE = 256 # (MODEL_SIZE, MODEL_SIZE, 1) a.k.a (256, 256, 1) 
BATCH_SIZE = 32

# Resize Image using Tensorflow fast and not OOM. 
rsz = lambda x: tf.image.resize(x, (MODEL_SIZE,MODEL_SIZE), antialias=True)

train_dir = tf.data.Dataset.from_tensor_slices((rsz(X_train), y_train))
val_dir = tf.data.Dataset.from_tensor_slices((rsz(X_valid), y_valid))
test_dir = tf.data.Dataset.from_tensor_slices((rsz(X_test), y_test))

# if i use map(resizing) result OOM
train_dir = train_dir.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
val_dir = val_dir.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)
test_dir = test_dir.batch(BATCH_SIZE)

In [ ]:
del X_train, y_train, X_valid, y_valid, X_test, y_test

In [ ]:
################################################## 30 SRM FILTERS
srm_weights = np.load('../input/stegan/SRM_Kernels.npy') 
biasSRM= np.ones(30)
print (srm_weights.shape)
################################################## TLU ACTIVATION FUNCTION
T3 = 3;
def Tanh3(x):
    tanh3 = tf.keras.activations.tanh(x)*T3
    return tanh3
##################################################

In [ ]:
class SppnetLayer(tf.keras.layers.Layer):
    def __init__(self, pool_sizes=[1, 2, 4], **kwargs):
        super(SppnetLayer, self).__init__(**kwargs)
        self.pool_sizes = pool_sizes

    def call(self, inputs):
        pooled_outputs = []
        for pool_size in self.pool_sizes:
            pool = tf.keras.layers.MaxPooling2D(
                pool_size=(pool_size, pool_size),
                strides=(pool_size, pool_size),
                padding="same"
            )(inputs)
            pooled_outputs.append(tf.keras.layers.Flatten()(pool))
        return tf.concat(pooled_outputs, axis=-1)

In [ ]:
from tensorflow.keras import backend as K
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Activation

def Tanh3(x):
    return tf.tanh(3.0 * x)

def modelStegan(
    input_shape=(MODEL_SIZE, MODEL_SIZE, 1),
    lr=0.00001,
    initial_dropout_rate=0.2, # Reduced slightly from 0.3 for less aggressive regularization
    l2_penalty=1e-5,          # Reduced from 1e-4 for less aggressive regularization
    bn_momentum=0.9,          # Keeping this at 0.9 as it's more standard for stable BN
    srm_kernels_path='/kaggle/input/stegan/SRM_Kernels.npy',
    biasSRM=np.zeros(30, dtype=np.float32),
):
    tf.keras.backend.clear_session()
    
    # Load SRM weights (only once per model instance creation)
    try:
        srm_weights = np.load(srm_kernels_path)
    except FileNotFoundError:
        print(f"Error: SRM kernels not found at {srm_kernels_path}. Please check the path.")
        return None # Or handle error appropriately

    kernel_reg = regularizers.l2(l2_penalty) if l2_penalty > 0 else None
    inputs = tf.keras.Input(shape=input_shape, name="input_1")

    # --- Data Augmentation (from v4) ---
    x = tf.keras.layers.RandomFlip("horizontal_and_vertical", name="random_flip")(inputs)

    # --- Layer 1: Fixed SRM Filters (from V1 and V4 fixed_srm) ---
    # This will be the first block focusing on initial noise extraction
    fixed_srm_1 = tf.keras.layers.Conv2D(
        30, (5, 5), strides=(1, 1), padding='same',
        activation=Tanh3, use_bias=True,
        kernel_initializer=tf.keras.initializers.Constant(srm_weights),
        bias_initializer=tf.keras.initializers.Constant(biasSRM),
        trainable=False, name="fixed_srm_block_1"
    )(x)
    x = tf.keras.layers.BatchNormalization(
        momentum=bn_momentum, epsilon=0.001, center=True, scale=True # Keeping scale=True for standard BN
    )(fixed_srm_1)
    # No explicit activation after BN for this initial part, will be handled by subsequent layers
    
    # --- Layer 2: Second Block of Fixed SRM Filters (NEW - to capture more diverse noise) ---
    # Use different set of SRM kernels if available, or apply the same set again for deeper feature.
    # For simplicity, using the same SRM kernels, but applying them to the original input
    # or a lightly processed version for different initial feature space.
    # Here, let's concatenate another fixed SRM to capture features from the initial input,
    # doubling the initial feature space with similar, yet distinct, filtering.
    fixed_srm_2 = tf.keras.layers.Conv2D(
        30, (5, 5), strides=(1, 1), padding='same',
        activation=Tanh3, use_bias=True,
        kernel_initializer=tf.keras.initializers.Constant(srm_weights),
        bias_initializer=tf.keras.initializers.Constant(biasSRM),
        trainable=False, name="fixed_srm_block_2"
    )(inputs) # Apply to original input to extract different set of base features
    fixed_srm_2_bn = tf.keras.layers.BatchNormalization(
        momentum=bn_momentum, epsilon=0.001, center=True, scale=True
    )(fixed_srm_2)

    # Concatenate the outputs from both fixed SRM blocks (60 channels total)
    x = tf.keras.layers.Concatenate(axis=-1, name="srm_combined")([x, fixed_srm_2_bn])
    
    # Apply initial activation after combined SRM features
    x = Activation("elu")(x) # This serves as the 'layers1_output' for the first residual block

    # --- Residual Block 1 (Adapted from V4's structure but with V1's successful regularization strategy and channel counts) ---
    # Keeping the depthwise + Conv2D structure as it's common and good.
    # Channels will increase more gradually.
    
    # Skip connection (from v4's approach for the residual path, but matching v1's channel focus)
    res1_skip = tf.keras.layers.Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(x) # Use 1x1 conv to match channels for skip
    res1_skip = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res1_skip)
    res1_skip = Activation("elu")(res1_skip) # Activation after skip connection

    # Main path
    res1_main = tf.keras.layers.DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(x)
    res1_main = tf.keras.layers.Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res1_main) # Pointwise conv after depthwise
    res1_main = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res1_main)
    res1_main = Activation("elu")(res1_main)

    res1_main = tf.keras.layers.DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(res1_main)
    res1_main = tf.keras.layers.Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res1_main) # Pointwise conv after depthwise
    res1_main = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res1_main)
    res1_main = Activation("elu")(res1_main)

    x = tf.keras.layers.Add()([res1_skip, res1_main])

    # --- Conv Layers after Residual Block 1 ---
    x = tf.keras.layers.Conv2D(60, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    
    x = tf.keras.layers.Conv2D(60, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = tf.keras.layers.AveragePooling2D((2, 2))(x)

    # --- Residual Block 2 ---
    # Channels potentially increase here to 120 or remain 60, let's increase for deeper features
    # Keeping 60 for now, might be better to increase later if needed
    
    res2_skip = tf.keras.layers.Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(x)
    res2_skip = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res2_skip)
    res2_skip = Activation("elu")(res2_skip)

    res2_main = tf.keras.layers.DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(x)
    res2_main = tf.keras.layers.Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res2_main)
    res2_main = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res2_main)
    res2_main = Activation("elu")(res2_main)

    res2_main = tf.keras.layers.DepthwiseConv2D((3, 3), padding='same', depthwise_regularizer=kernel_reg)(res2_main)
    res2_main = tf.keras.layers.Conv2D(60, (1, 1), padding='same', kernel_regularizer=kernel_reg)(res2_main)
    res2_main = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(res2_main)
    res2_main = Activation("elu")(res2_main)

    x = tf.keras.layers.Add()([res2_skip, res2_main])
    
    # Conv Layer after ResBlock2
    x = tf.keras.layers.Conv2D(60, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x)
    x = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = tf.keras.layers.AveragePooling2D((2, 2))(x)

    # Downsample block (can be adjusted for more channels as feature maps get smaller)
    x = tf.keras.layers.Conv2D(90, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x) # Increased channels here
    x = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = tf.keras.layers.AveragePooling2D((2, 2))(x)

    x = tf.keras.layers.Conv2D(120, (3, 3), padding='same', kernel_regularizer=kernel_reg)(x) # Increased channels further
    x = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)
    x = tf.keras.layers.AveragePooling2D((2, 2))(x)

    # SPPF Block
    sppf_input = x
    sppf_branches = [sppf_input]
    
    for pool_size_val in [5, 7]: # Using 5 and 7 is typical
        pooled = tf.keras.layers.MaxPooling2D(
            pool_size=pool_size_val, strides=1, padding='same'
        )(sppf_input)
        sppf_branches.append(pooled)

    x = tf.keras.layers.Concatenate(axis=-1)(sppf_branches) # Channels will be 120 * 3 = 360
    
    # Conv after SPPF concatenation
    x = tf.keras.layers.Conv2D(120, (1, 1), padding='same', kernel_regularizer=kernel_reg)(x) # Reduce channels
    x = tf.keras.layers.BatchNormalization(momentum=bn_momentum, epsilon=0.001, center=True, scale=True)(x)
    x = Activation("elu")(x)

    # Dropout Point 1 (After SPPF) - Retaining strategic dropout from v4
    if initial_dropout_rate > 0:
        x = tf.keras.layers.Dropout(initial_dropout_rate, name="dropout_after_sppf")(x)

    # MHA Block
    def mha_block(tensor, num_heads=4):
        h, w, c = tensor.shape[1], tensor.shape[2], tensor.shape[3]
        if None in (h, w, c):
             # Fallback to dynamic shape if static not available (e.g., from tf.data pipeline)
            h, w, c = K.int_shape(tensor)[1], K.int_shape(tensor)[2], K.int_shape(tensor)[3]
            if None in (h, w, c): # If still None, something is wrong, raise error
                raise ValueError("MHA block: Dynamic spatial or channel dimensions encountered and could not be inferred.")

        key_dim = c // num_heads
        if key_dim == 0:
            print(f"Warning: key_dim for MHA is {key_dim} (channels: {c}, num_heads: {num_heads}). Setting to 1.")
            key_dim = 1 # Fallback to prevent error, but ideally avoid this if channels are very low

        reshaped = tf.keras.layers.Reshape((-1, c))(tensor)
        mha_layer = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        attention_output = mha_layer(reshaped, reshaped)
        mha_reshaped = tf.keras.layers.Reshape((h, w, c))(attention_output)
        out = tf.keras.layers.Add()([tensor, mha_reshaped])
        out = tf.keras.layers.LayerNormalization(epsilon=1e-6)(out)
        return out

    x = mha_block(x)

    # Dropout Point 2 (After MHA)
    if initial_dropout_rate > 0:
        x = tf.keras.layers.Dropout(initial_dropout_rate, name="dropout_after_mha")(x)
    
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x) # Keeping for MHA stability

    # Output layers
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    
    # Dropout Point 3 (Before Dense)
    if initial_dropout_rate > 0:
        x = tf.keras.layers.Dropout(initial_dropout_rate, name="dropout_before_dense")(x)
    
    outputs = tf.keras.layers.Dense(2, activation='softmax', kernel_regularizer=kernel_reg)(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=lr), # Reverting to AdamW with softer regularization
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
model = modelStegan()
model.summary()

In [ ]:
# model = modelStegan(img_size=img_size)
historyRSNet = model.fit(train_dir,
                         epochs=100,
                         batch_size=32,
                         validation_data=val_dir,
                         #   callbacks=[reduce_lr]
)

In [ ]:
plt.plot(historyRSNet.history['accuracy'])
plt.plot(historyRSNet.history['val_accuracy'])

In [ ]:
plt.plot(historyRSNet.history['loss'])
plt.plot(historyRSNet.history['val_loss'])

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

def show_report(y_pred, y_true):
		cm = confusion_matrix(y_pred, y_true)
		print(cm)
		print(classification_report(y_pred, y_true, digits=4))

In [ ]:
print("Confusion Matrix Data Test")
y_pred = model.predict(train_dir)
predicted_categories = tf.argmax(y_pred, axis=1)
true_categories = tf.cast([np.argmax(y) for _, y in train_dir for y in y],dtype=np.int64)

show_report(predicted_categories, true_categories)

In [ ]:
model.save('SteganBossUni51251202.keras')